## 1. Setup

In [ ]:
#notebook executed but results cleaned, progress plots can be found in evaluation notebook

import torch, time, os, glob, numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import MultiStepLR
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

import importlib, subprocess, sys
for pkg,pip in [("skimage","scikit-image"),("cv2","opencv-python-headless"),("matplotlib","matplotlib")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable,"-m","pip","install","-q",pip])
from skimage.metrics import peak_signal_noise_ratio as psnr
import cv2, matplotlib.pyplot as plt
print("deps ready")

## 2. Configuration


In [ ]:
DEPTH        = 20            # DnCNN-B
PATCH_SIZE   = 50           # blind uses 50x50
STRIDE       = 10
SCALES       = [1, 0.9, 0.8, 0.7]
AUG_TIMES    = 1
BATCH_SIZE   = 128
LR           = 1e-3
MILESTONES   = [30, 60, 90]
GAMMA        = 0.2
EPOCHS       = 30           # matched what worked for DnCNN-S
SIGMA_MIN    = 0
SIGMA_MAX    = 55           # blind noise range [0, 55]

# --- Save checkpoints to Google Drive so they survive a disconnect ---
SAVE_TO_DRIVE = True        # set False to keep checkpoints only in the Colab session
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = "/content/drive/MyDrive/dncnn_b_blind"   # <-- folder in your Drive
        print("Checkpoints will be saved to Drive:", SAVE_DIR)
    except Exception as e:
        SAVE_DIR = "models_dncnn_b_blind"
        print("Drive mount unavailable (", e, ") -> saving locally to", SAVE_DIR)
else:
    SAVE_DIR = "models_dncnn_b_blind"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"DnCNN-B | depth={DEPTH}, patch={PATCH_SIZE}, sigma in [{SIGMA_MIN},{SIGMA_MAX}], "
      f"epochs={EPOCHS}\nSAVE_DIR = {SAVE_DIR}")

## 3. Training data — Train400 (same source as the S notebook)

In [ ]:
DATA_DIR = "data/Train400"
os.makedirs(DATA_DIR, exist_ok=True)
import urllib.request
BASE = ("https://raw.githubusercontent.com/cszn/DnCNN/master/"
        "TrainingCodes/DnCNN_TrainingCodes_v1.0/data/Train400")
if len(glob.glob(DATA_DIR + "/*.png")) < 400:
    print("Downloading Train400 ...")
    ok = fail = 0
    for n in range(1, 401):
        fn = f"test_{n:03d}.png"; dst = os.path.join(DATA_DIR, fn)
        if os.path.exists(dst): ok += 1; continue
        try: urllib.request.urlretrieve(f"{BASE}/{fn}", dst); ok += 1
        except Exception as e:
            fail += 1
            if fail <= 3: print("  fail", fn, e)
        if n % 100 == 0: print(f"  {n}/400 ...")
    print(f"done: {ok} ok, {fail} failed")
files = sorted(glob.glob(DATA_DIR + "/*.png"))
print("Train400 images:", len(files))
assert len(files) > 0, "No training images - upload Train400 PNGs to " + DATA_DIR

## 4. Patch extraction (50x50)


In [ ]:
def data_aug(img, mode=0):
    if mode==0: return img
    elif mode==1: return np.flipud(img)
    elif mode==2: return np.rot90(img)
    elif mode==3: return np.flipud(np.rot90(img))
    elif mode==4: return np.rot90(img, k=2)
    elif mode==5: return np.flipud(np.rot90(img, k=2))
    elif mode==6: return np.rot90(img, k=3)
    elif mode==7: return np.flipud(np.rot90(img, k=3))

def gen_patches(file_name):
    img = cv2.imread(file_name, 0)
    if img is None: return []
    h, w = img.shape; patches = []
    for s in SCALES:
        hs, ws = int(h*s), int(w*s)
        im = cv2.resize(img, (hs, ws), interpolation=cv2.INTER_CUBIC)
        for i in range(0, hs-PATCH_SIZE+1, STRIDE):
            for j in range(0, ws-PATCH_SIZE+1, STRIDE):
                x = im[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
                for _ in range(AUG_TIMES):
                    patches.append(data_aug(x, mode=np.random.randint(0,8)))
    return patches

def datagenerator(data_dir):
    data = []
    for fn in sorted(glob.glob(data_dir + "/*.png")):
        data.extend(gen_patches(fn))
    if len(data)==0:
        raise RuntimeError("No patches generated - check data/Train400 and PATCH_SIZE.")
    data = np.array(data, dtype="uint8")
    data = np.expand_dims(data, axis=3)
    discard = len(data) - len(data)//BATCH_SIZE*BATCH_SIZE
    data = np.delete(data, range(discard), axis=0)
    return data

xs = datagenerator(DATA_DIR)
print("patches:", xs.shape)

## 5. Blind dataset: random sigma per patch



In [ ]:
class BlindDenoisingDataset(Dataset):
    """Adds AWGN with a per-sample random sigma in [smin, smax] (0-255 scale)."""
    def __init__(self, xs, smin=0, smax=55):
        super().__init__()
        self.xs = xs; self.smin = smin; self.smax = smax
    def __getitem__(self, i):
        batch_x = self.xs[i]
        sigma = np.random.uniform(self.smin, self.smax)          # random level per patch
        noise = torch.randn(batch_x.size()).mul_(sigma/255.0)
        batch_y = batch_x + noise
        return batch_y, batch_x
    def __len__(self):
        return self.xs.size(0)

xs_t = torch.from_numpy(xs.astype("float32")/255.0).permute(0,3,1,2)
train_ds = BlindDenoisingDataset(xs_t, SIGMA_MIN, SIGMA_MAX)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, drop_last=True)
print("patches:", tuple(xs_t.shape), "| batches/epoch:", len(train_dl))

## 6. Model + loss: authors' code at depth 20



In [ ]:
import urllib.request, types
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/TrainingCodes/dncnn_pytorch"
for fn in ["main_train.py"]:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(f"{RAW}/{fn}", fn); print("downloaded", fn)

def import_authors_classes(path="main_train.py"):
    lines = open(path).read().splitlines()
    def idx(pred):
        for i,l in enumerate(lines):
            if pred(l): return i
        return -1
    i_arg   = idx(lambda l: l.strip().startswith("parser = argparse"))
    i_class = idx(lambda l: l.startswith("class "))
    i_main  = idx(lambda l: l.strip().startswith("if __name__"))
    head = lines[:i_arg] + [""] + lines[i_class:(i_main if i_main!=-1 else len(lines))]
    mod = types.ModuleType("authors_dncnn")
    exec(compile("\n".join(head), "authors_dncnn", "exec"), mod.__dict__)
    return mod

authors = import_authors_classes()
DnCNN = authors.DnCNN
sum_squared_error = authors.sum_squared_error

model = DnCNN(depth=DEPTH).to(device)      # depth=20 for blind
criterion = sum_squared_error()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
print(f"DnCNN-B depth={DEPTH} | params:", sum(p.numel() for p in model.parameters()))

## 7. Validation across noise levels


In [ ]:
import urllib.request
os.makedirs("Set12", exist_ok=True)
base = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/Set12"
val_imgs = []
for n in range(1,13):
    p = f"Set12/{n:02d}.png"
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(f"{base}/{n:02d}.png", p)
        except Exception as e: print("dl fail", n, e)
    if os.path.exists(p):
        val_imgs.append(cv2.imread(p,0).astype("float32")/255.0)
print("val images:", len(val_imgs))

@torch.no_grad()
def validate(net, sigmas=(15,25,50), seed=0):
    net.eval(); out = {}
    for sigma in sigmas:
        ps = []
        for k,clean in enumerate(val_imgs):
            rng = np.random.default_rng(seed+k)
            noisy = clean + rng.normal(0, sigma/255.0, clean.shape)
            t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)
            den = net(t).squeeze().cpu().numpy().clip(0,1)
            ps.append(psnr(clean, den, data_range=1.0))
        out[sigma] = float(np.mean(ps))
    net.train(); return out

print("PSNR before training:", {k:round(v,2) for k,v in validate(model).items()})

## 8. Training loop (resume-aware)


In [ ]:
import re

# --- Resume support: pick up from the latest checkpoint in SAVE_DIR if present ---
def latest_checkpoint(save_dir):
    cks = glob.glob(os.path.join(save_dir, "model_*.pth"))
    if not cks: return None, 0
    def epnum(p):
        m = re.search(r"model_(\d+)\.pth$", p); return int(m.group(1)) if m else 0
    latest = max(cks, key=epnum)
    return latest, epnum(latest)

ckpt, start_epoch = latest_checkpoint(SAVE_DIR)
if ckpt is not None:
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print(f"Resuming from {ckpt} (completed epoch {start_epoch}).")
    print("Note: model weights restored; Adam/scheduler state restarts "
          "(small transient, negligible over many epochs).")
else:
    start_epoch = 0
    print("No checkpoint found - training from scratch.")

history = globals().get("history", [])   # keep history across reconnects if still in memory

for epoch in range(start_epoch, EPOCHS):
    scheduler.step(epoch)
    model.train()
    t0 = time.time(); running = 0.0; nb_ = 0
    for batch_y, batch_x in train_dl:
        batch_y, batch_x = batch_y.to(device), batch_x.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_y), batch_x)
        loss.backward(); optimizer.step()
        running += loss.item(); nb_ += 1
    v = validate(model)
    history.append({"epoch": epoch+1, "loss": running/nb_/BATCH_SIZE, **{f"psnr_{k}":v[k] for k in v}})
    print(f"epoch {epoch+1:3d}/{EPOCHS} | loss {running/nb_/BATCH_SIZE:8.4f} "
          f"| PSNR  15:{v[15]:.2f}  25:{v[25]:.2f}  50:{v[50]:.2f} dB | {time.time()-t0:.1f}s")
    # save to SAVE_DIR (Drive) every epoch so a disconnect loses at most one epoch
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"model_{epoch+1:03d}.pth"))
print("done. checkpoints in", SAVE_DIR)